# Stress Prediction V53 — Joint Tree/Pair Quantiles

Train 전용 개발·감사 탐색에서 선택된 Tree 54%, Pair 48%, Pair weight 28% 설정입니다. 5-Fold MAE와 제출 파일을 생성합니다.


In [ ]:
from __future__ import annotations

import argparse
import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.52
PAIR_WEIGHT = 0.15

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant", fill_value="missing"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_quantile_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_test = add_feature_copies(test_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    random_state=RANDOM_SEED,
                    n_jobs=1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    matrix = pipeline.named_steps["preprocessor"].transform(weighted_test)
    forest = pipeline.named_steps["model"]
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in forest.estimators_]
    )
    return np.clip(
        np.round(np.quantile(tree_predictions, TREE_QUANTILE, axis=1), 2),
        0.0,
        1.0,
    )


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Train 분포만으로 Train과 한 개씩 독립적인 Test 값을 변환."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(
        sorted_train, train_values, side="right"
    ) / len(train_values)
    test_rank = np.searchsorted(
        sorted_train, test_values, side="right"
    ) / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """각 2차원 피처쌍에서 가장 가까운 Train target을 분위수 집계."""
    train_rank = {}
    test_rank = {}
    for column in PAIR_COLUMNS:
        train_rank[column], test_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            test_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(test_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(test_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_index = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_index])
    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def validate_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")

In [ ]:
from sklearn.model_selection import KFold

DATA_DIR=Path('/Users/bitsaem/Desktop/stress_project/open'); OUTPUT_DIR=Path('/Users/bitsaem/Desktop/stress_project')
train=pd.read_csv(DATA_DIR/'train.csv'); test=pd.read_csv(DATA_DIR/'test.csv'); submission=pd.read_csv(DATA_DIR/'sample_submission.csv'); validate_inputs(train,test,submission)
X=add_features(train.drop(columns=[TARGET,ID_COLUMN])).reset_index(drop=True); X_test=add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True); y=train[TARGET].reset_index(drop=True)
TREE_Q=.54; PAIR_Q=.54; PAIR_W=1.00; PAIRS=list(combinations(PAIR_COLUMNS,2))

def tree_q_prediction(a,b,t):
    aa=add_feature_copies(a); bb=add_feature_copies(b); pre=make_preprocessor(aa); am=pre.fit_transform(aa); bm=pre.transform(bb)
    model=ExtraTreesRegressor(n_estimators=1200,min_samples_leaf=1,max_features=1,random_state=42,n_jobs=-1); model.fit(am,t)
    matrix=np.column_stack([tree.predict(bm) for tree in model.estimators_])
    return np.quantile(matrix,TREE_Q,axis=1)

def pair_q_prediction(a,b,t):
    tr,va={},{}
    for c in PAIR_COLUMNS: tr[c],va[c]=empirical_rank_transform(a[c].to_numpy(float),b[c].to_numpy(float))
    tv=t.to_numpy(float); values=[]
    for c1,c2 in PAIRS:
        distance=np.abs(va[c1][:,None]-tr[c1][None,:])+np.abs(va[c2][:,None]-tr[c2][None,:])
        values.append(tv[np.argmin(distance,axis=1)])
    return np.quantile(np.column_stack(values),PAIR_Q,axis=1)

def predict_v53(a,b,t):
    tree=tree_q_prediction(a,b,t); pair=pair_q_prediction(a,b,t)
    return np.clip(np.round((1-PAIR_W)*tree+PAIR_W*pair,2),0,1)

cv=KFold(5,shuffle=True,random_state=42); oof=np.zeros(len(y)); rows=[]
for fold,(ti,vi) in enumerate(cv.split(X),1):
    oof[vi]=predict_v53(X.iloc[ti],X.iloc[vi],y.iloc[ti]); score=mean_absolute_error(y.iloc[vi],oof[vi]); rows.append({'Fold':fold,'V53_MAE':score}); print(f'Fold {fold} V53 MAE: {score:.6f}')
print('V53 CV MAE:',f'{mean_absolute_error(y,oof):.6f}'); display(pd.DataFrame(rows))

prediction=predict_v53(X,X_test,y); output=submission.copy(); output[TARGET]=prediction
path=OUTPUT_DIR/'submit_v53_pair_only_q54.csv'; output.to_csv(path,index=False,encoding='utf-8'); print('Saved:',path); print('Prediction range:',prediction.min(),prediction.max()); display(output.head(10))



## Near-Full 검증

실제 제출 학습량에 가까운 2,850명 학습 / 150명 검증을 8회 반복하여 기존 모델과 비교합니다.


In [ ]:
import json
from pathlib import Path
base=json.loads(Path('/Users/bitsaem/Desktop/stress_project/stress_prediction_v53_pair_only_loo_q54.ipynb').read_text())
exec(''.join(base['cells'][1]['source']),globals())
from sklearn.model_selection import ShuffleSplit

DATA=Path('/Users/bitsaem/Desktop/stress_project/open'); train=pd.read_csv(DATA/'train.csv')
X=add_features(train.drop(columns=[TARGET,ID_COLUMN])).reset_index(drop=True); y=train[TARGET].reset_index(drop=True)
PAIRS=list(combinations(PAIR_COLUMNS,2))
def tq(a,b,t,q):
 aa=add_feature_copies(a); bb=add_feature_copies(b); pre=make_preprocessor(aa); am=pre.fit_transform(aa); bm=pre.transform(bb); model=ExtraTreesRegressor(n_estimators=1200,min_samples_leaf=1,max_features=1,random_state=42,n_jobs=-1); model.fit(am,t); m=np.column_stack([z.predict(bm) for z in model.estimators_]); return np.quantile(m,q,axis=1)
def pm(a,b,t):
 tr,va={},{}
 for c in PAIR_COLUMNS: tr[c],va[c]=empirical_rank_transform(a[c].to_numpy(float),b[c].to_numpy(float))
 tv=t.to_numpy(float); out=[]
 for c1,c2 in PAIRS:
  d=np.abs(va[c1][:,None]-tr[c1][None,:])+np.abs(va[c2][:,None]-tr[c2][None,:]); out.append(tv[np.argmin(d,axis=1)])
 return np.column_stack(out)
rows=[]; cv=ShuffleSplit(n_splits=8,test_size=150,random_state=29082026)
for s,(ti,vi) in enumerate(cv.split(X),1):
 a,b,t,v=X.iloc[ti],X.iloc[vi],y.iloc[ti],y.iloc[vi]; pair=pm(a,b,t); tree52=tq(a,b,t,.52); tree54=tq(a,b,t,.54)
 basep=np.clip(np.round(.85*tree52+.15*np.quantile(pair,.52,axis=1),2),0,1); v53=np.clip(np.round(.72*tree54+.28*np.quantile(pair,.48,axis=1),2),0,1)
 bm=mean_absolute_error(v,basep); cm=mean_absolute_error(v,v53); rows.append(dict(split=s,baseline_MAE=bm,V53_MAE=cm,improvement=bm-cm)); print(s,f'base={bm:.6f}',f'v53={cm:.6f}',flush=True)
df=pd.DataFrame(rows); print(df); print('Baseline Near-Full MAE:',df.baseline_MAE.mean()); print('V53 Near-Full MAE:',df.V53_MAE.mean()); print('Mean improvement:',df.improvement.mean()); print('Wins:',int((df.improvement>0).sum()),'/ 8')

